<a href="https://colab.research.google.com/github/Nafeesatu/API-Calls/blob/main/First_API_Calls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 6 — Notebook 1: First API Calls
## Anthropic + Groq side by side

**What you'll learn:**
- How to install and import LLM client libraries
- How to load API keys safely (never hardcode them!)
- The anatomy of an API call: messages, temperature, max_tokens
- How to read token usage from the response
- Why two top models give similar but different answers

---
> **Before you start:** Add your API keys to Colab Secrets (🔑 icon in the left sidebar).
> Name them exactly: `ANTHROPIC_API_KEY` and `GROQ_API_KEY`

## Step 1 — Install the packages

We install the packages for Anthropic, Gemini, and Groq:
- `anthropic` — the official Python client for Anthropic's API
- `groq` — the official Python client for Groq's API

The `-q` flag means 'quiet' — it suppresses the installation output.

In [1]:
!pip install anthropic google-generativeai groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.3 MB/s eta 0:00:00


## Step 2 — Import libraries and load API keys

We use `google.colab.userdata` to load keys from Colab Secrets.

**Why not just paste the key in?** If you share or commit this notebook, your key is exposed. Using Secrets means the key never appears in the notebook itself.

In [2]:
import anthropic
import groq
from google.colab import userdata



In [3]:
# Load keys from Colab Secrets
# As per user's request, only loading keys for Anthropic, Gemini, and Groq.
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
GROK_API_KEY     = userdata.get('GROK')

# Confirm they loaded (we only print the first 8 chars for safety)
print(f"Anthropic key loaded: {ANTHROPIC_API_KEY[:8]}...")
print(f"Grok key loaded:      {GROK_API_KEY[:8]}...")

Anthropic key loaded: sk-ant-a...
Grok key loaded:      gsk_jFN1...


## Step 3 — Create the API clients

Each provider gives you a **client object**. Think of it as your "phone" to that provider.
All calls go through this client.

In [5]:
# Create clients — these objects handle authentication automatically
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
groq_client      = groq.Groq(api_key=GROK_API_KEY)

print("Anthropic, Gemini, and Groq clients created successfully!")

Anthropic, Gemini, and Groq clients created successfully!


 ## Step 4 — Your first Anthropic (Claude) call

### Anthropic's API is slightly different:

```python
anthropic_client.messages.create(
    model=...,       # Which Claude model
    max_tokens=...,  # REQUIRED in Anthropic (unlike OpenAI)
    system=...,      # System prompt is a separate parameter (not in messages list)
    messages=[...]   # Only user/assistant turns go here
)
```

Key difference: In Anthropic, `system` is a top-level parameter, not inside the `messages` list.

In [15]:
QUESTION = "Explain what a neural network is in exactly 3 sentences, for a complete beginner."

# --- ANTHROPIC CALL ---
# anthropic_response = anthropic_client.messages.create(
#     model="claude-haiku-4-5",     # Fast, cheap Claude model — equivalent tier to gpt-4o-mini
#     max_tokens=200,               # Required in Anthropic's API
#     system="You are a helpful assistant who explains technical concepts simply.",
#     messages=[
#         {
#             "role": "user",
#             "content": QUESTION,
#         }
#     ]
# )

# # Extract the text
# anthropic_text = anthropic_response.content[0].text

# # Extract token usage
# anthropic_tokens = anthropic_response.usage

print("=" * 60)
print("ANTHROPIC (Claude Haiku) RESPONSE:")
print("=" * 60)
# print(anthropic_text)
print("Anthropic response not available due to API error: Insufficient credit.")
print()
# print(f"Tokens used — Input: {anthropic_tokens.input_tokens} | Output: {anthropic_tokens.output_tokens}")

ANTHROPIC (Claude Haiku) RESPONSE:
Anthropic response not available due to API error: Insufficient credit.



In [8]:
QUESTION = "Explain what a neural network is in exactly 3 sentences, for a complete beginner."

# --- GROQ CALL ---
groq_response = groq_client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant who explains technical concepts simply."
        },
        {
            "role": "user",
            "content": QUESTION,
        }
    ],
    model="llama-3.1-8b-instant", # Updated to a currently supported model
    temperature=0.7,
    max_tokens=200,
)

# Extract the text
groq_text = groq_response.choices[0].message.content

# Extract token usage
groq_tokens = groq_response.usage

print("=" * 60)
print("GROQ (llama-3.1-8b-instant) RESPONSE:")
print("=" * 60)
print(groq_text)
print()
print(f"Tokens used — Input: {groq_tokens.prompt_tokens} | Output: {groq_tokens.completion_tokens}")

GROQ (llama-3.1-8b-instant) RESPONSE:
A neural network is a computer system that's inspired by the way our brains work. It's made up of many small units called "neurons" that are connected together and work together to recognize patterns in data, such as pictures or sounds. By giving the network lots of examples of what it should be recognizing, it can learn to make its own decisions and improve its performance over time.

Tokens used — Input: 64 | Output: 79


In [9]:
# List available Groq models
print("Available Groq Models:")
for model in groq_client.models.list().data:
    print(f"- {model.id}")

Available Groq Models:
- qwen/qwen3-32b
- whisper-large-v3-turbo
- llama-3.3-70b-versatile
- meta-llama/llama-prompt-guard-2-22m
- llama-3.1-8b-instant
- groq/compound
- openai/gpt-oss-120b
- allam-2-7b
- openai/gpt-oss-20b
- qwen/qwen3.6-27b
- canopylabs/orpheus-arabic-saudi
- whisper-large-v3
- meta-llama/llama-4-scout-17b-16e-instruct
- groq/compound-mini
- openai/gpt-oss-safeguard-20b
- canopylabs/orpheus-v1-english
- meta-llama/llama-prompt-guard-2-86m


## Step 6 — Side-by-side comparison

In [11]:
print("QUESTION:", QUESTION)
print()
print("━" * 70)
print("ANTHROPIC (claude-haiku-4-5) RESPONSE:")
print("━" * 70)
# print(anthropic_text) # Commented out due to previous Anthropic API error
print("  Anthropic response not available due to API error: Insufficient credit.")
print()
# print(f"  → Tokens: {anthropic_tokens.input_tokens} input | {anthropic_tokens.output_tokens} output | Total: {anthropic_tokens.input_tokens + anthropic_tokens.output_tokens} total") # Commented out due to previous Anthropic API error
print()
print("━" * 70)
print("GROQ (llama-3.1-8b-instant) RESPONSE:")
print("━" * 70)
print(groq_text)
print()
print(f"  → Tokens: {groq_tokens.prompt_tokens} input | {groq_tokens.completion_tokens} output | Total: {groq_tokens.prompt_tokens + groq_tokens.completion_tokens} total")
print()
print("━" * 70)
print("OBSERVATIONS:")
print("  - Both models provided an answer to the question.")
print("  - Token counts indicate the length of the input and output.")

QUESTION: Explain what a neural network is in exactly 3 sentences, for a complete beginner.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ANTHROPIC (claude-haiku-4-5) RESPONSE:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Anthropic response not available due to API error: Insufficient credit.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GROQ (llama-3.1-8b-instant) RESPONSE:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A neural network is a computer system that's inspired by the way our brains work. It's made up of many small units called "neurons" that are connected together and work together to recognize patterns in data, such as pictures or sounds. By giving the network lots of examples of what it should be recognizing, it can learn to make its own decisions and improve its performance over time.

  → Tokens: 64 input | 79 output | Total: 143 total

━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 7
### Experiment: What does temperature ACTUALLY do for Anthropic and Groq?

`temperature` is a parameter in LLM APIs that controls the *randomness* of the output.

- **Lower temperature (e.g., 0.0):** The model will choose the most probable words, resulting in more deterministic, focused, and repeatable output.
- **Higher temperature (e.g., 1.0):** The model will consider a wider range of less probable words, leading to more diverse, creative, and sometimes surprising output

#### Anthropic Temperature Experiment

In [13]:
print("--- Anthropic with temperature = 0.0 ---")
# anthropic_response_temp_0 = anthropic_client.messages.create(
#     model="claude-haiku-4-5",
#     max_tokens=200,
#     system="You are a helpful assistant who explains technical concepts simply.",
#     messages=[
#         {
#             "role": "user",
#             "content": QUESTION,
#         }
#     ],
#     temperature=0.0 # Low temperature
# )
# print(anthropic_response_temp_0.content[0].text)
print("  Anthropic response not available due to API error: Insufficient credit.")

print("\n--- Anthropic with temperature = 1.0 ---")
# anthropic_response_temp_1 = anthropic_client.messages.create(
#     model="claude-haiku-4-5",
#     max_tokens=200,
#     system="You are a helpful assistant who explains technical concepts simply.",
#     messages=[
#         {
#             "role": "user",
#             "content": QUESTION,
#         }
#     ],
#     temperature=1.0 # High temperature
# )
# print(anthropic_response_temp_1.content[0].text)
print("  Anthropic response not available due to API error: Insufficient credit.")

--- Anthropic with temperature = 0.0 ---
  Anthropic response not available due to API error: Insufficient credit.

--- Anthropic with temperature = 1.0 ---
  Anthropic response not available due to API error: Insufficient credit.


#### Groq Temperature Experiment

In [14]:
print("\n--- Groq with temperature = 0.0 ---")
groq_response_temp_0 = groq_client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant who explains technical concepts simply."
        },
        {
            "role": "user",
            "content": QUESTION,
        }
    ],
    model="llama-3.1-8b-instant",
    temperature=0.0 # Low temperature
)
print(groq_response_temp_0.choices[0].message.content)

print("\n--- Groq with temperature = 1.0 ---")
groq_response_temp_1 = groq_client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant who explains technical concepts simply."
        },
        {
            "role": "user",
            "content": QUESTION,
        }
    ],
    model="llama-3.1-8b-instant",
    temperature=1.0 # High temperature
)
print(groq_response_temp_1.choices[0].message.content)


--- Groq with temperature = 0.0 ---
A neural network is a computer system that's inspired by the way our brains work. It's made up of many small processing units called "neurons" that are connected together, allowing them to share information and learn from it. Just like how our brains can recognize patterns and make decisions based on what we've learned, a neural network can be trained to recognize patterns in data and make predictions or decisions based on that information.

--- Groq with temperature = 1.0 ---
Here's a simple explanation: A neural network is a type of computer program that's inspired by the way our brains work. It's made up of many simple "nodes" or "neurons" that are connected together, and these nodes work together to help the computer make predictions or decisions based on the data it's given. Think of it like a complex web of interconnected thinking nodes that work together to figure things out, just like how your brain processes information.
